# Path E v2 — corrected ChEMBL Mycoplasma activity query

Companion to `notebooks/toxicity_path_e_mic_lookup.ipynb`. The v1 notebook's `assay_organism__icontains=Mycoplasma` filter was silently ignored by the ChEMBL REST API and returned 1088 unfiltered records (0 of which were actually Mycoplasma). v2 uses the correct query chain:

1. **UniProt accession → ChEMBL target ID** via `/target.json?target_components__accession=<UNIPROT>`. If a Mycoplasma protein has no ChEMBL target record, that's an honest absence — only proteins used in published bioassays show up in ChEMBL's target table.
2. **(target_chembl_id, molecule_chembl_id) → activities** via `/activity.json?target_chembl_id=...&molecule_chembl_id=...`. Returns ONLY assays of the specific compound on the specific Mycoplasma target.
3. **Fallback / universe check:** also query `/target.json?organism__icontains=Mycoplasma` to enumerate the complete set of Mycoplasma proteins represented in ChEMBL — gives a true ceiling on what's queryable.

Inputs: the 5 Mycoplasma UniProt orthologs found by v1's Layer 1 (P47587 / Q6MT28 / P75258 ileRS, P47508 / Q6MSR0 / P75398 leuRS, P47267 / P75091 metRS, P47252 / Q6MUI5 / P75106 tmk, P47359 / Q6MUF2 / P75521 asnRS) cross-referenced against the 4 resolved compound ChEMBL IDs (CHEMBL719 Mupirocin, CHEMBL443052 Tavaborole, CHEMBL4297370 REP3123, CHEMBL1366 Auranofin).

Output: `outputs/toxicity/path_e_mic_lookup_v2.json`. Auto-pushed at the end. Wall: ~3-5 minutes.

In [ ]:
# Cell 1 — install + clone + PAT prompt.
!pip install -q requests>=2.31
import os, subprocess, getpass
BRANCH = "claude/syn3a-whole-cell-simulator-REjHC"
REPO_URL = "https://github.com/Nikku03/cell.git"
REPO_DIR = "/content/cell"

def _run(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.stdout.strip(): print(r.stdout.rstrip())
    if r.stderr.strip(): print(r.stderr.rstrip())
    if r.returncode != 0: raise RuntimeError(f"{cmd!r} exit {r.returncode}")
    return r

if not os.path.isdir(REPO_DIR):
    _run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
else:
    _run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    _run(["git", "checkout", BRANCH], cwd=REPO_DIR)
    _run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_DIR)
%cd /content/cell

if not os.environ.get("GITHUB_PAT", "").strip():
    pat = getpass.getpass("Paste your GitHub PAT (input hidden): ").strip()
    if not pat: raise ValueError("empty PAT")
    os.environ["GITHUB_PAT"] = pat
print(f"PAT set ({len(os.environ['GITHUB_PAT'])} chars)")

In [ ]:
# Cell 2 — corrected target/compound matrix.
# Each row = (Syn3A locus, Mycoplasma UniProt orthologs from v1, canonical drug).
TARGETS = [
    {"locus": "JCVISYN3A_0519", "gene": "ileRS",
     "orthologs": [("P47587", "M. genitalium G37"),
                    ("Q6MT28", "M. mycoides SC"),
                    ("P75258", "M. pneumoniae M129")],
     "drug_chembl_id": "CHEMBL719", "drug_name": "Mupirocin"},
    {"locus": "JCVISYN3A_0634", "gene": "leuRS",
     "orthologs": [("P47508", "M. genitalium G37"),
                    ("Q6MSR0", "M. mycoides SC"),
                    ("P75398", "M. pneumoniae M129")],
     "drug_chembl_id": "CHEMBL443052", "drug_name": "Tavaborole"},
    {"locus": "JCVISYN3A_0012", "gene": "metRS",
     "orthologs": [("P47267", "M. genitalium G37"),
                    ("P75091", "M. pneumoniae M129")],
     "drug_chembl_id": "CHEMBL4297370", "drug_name": "REP3123"},
    {"locus": "JCVISYN3A_0045", "gene": "tmk",
     "orthologs": [("P47252", "M. genitalium G37"),
                    ("Q6MUI5", "M. mycoides SC"),
                    ("P75106", "M. pneumoniae M129")],
     "drug_chembl_id": None, "drug_name": "5-BrdUMP (research-stage; not in ChEMBL)"},
    {"locus": "JCVISYN3A_0076", "gene": "asnRS",
     "orthologs": [("P47359", "M. genitalium G37"),
                    ("Q6MUF2", "M. mycoides SC"),
                    ("P75521", "M. pneumoniae M129")],
     "drug_chembl_id": None, "drug_name": "Tirandamycin (research-stage; not in ChEMBL)"},
    {"locus": "JCVISYN3A_0819", "gene": "trxB",
     "orthologs": [],
     "drug_chembl_id": "CHEMBL1366", "drug_name": "Auranofin"},
]
for t in TARGETS:
    print(f"  {t['locus']}  {t['gene']:6s}  drug={t['drug_name'][:40]:40s}  orthologs={len(t['orthologs'])}")

In [ ]:
# Cell 3 — Layer 1: UniProt accession → ChEMBL target ID.
# For each Mycoplasma ortholog, query ChEMBL's target endpoint by
# UniProt accession. If no target record exists, ChEMBL has no
# bioassay history for that protein.
import requests, time, json
UA = {"User-Agent": "cell-sim-bot/1.0", "Accept": "application/json"}
CHEMBL = "https://www.ebi.ac.uk/chembl/api/data"

def chembl_target_for_uniprot(uniprot_acc):
    r = requests.get(f"{CHEMBL}/target.json",
                     params={"target_components__accession": uniprot_acc, "limit": "5"},
                     headers=UA, timeout=30)
    if r.status_code != 200:
        return None
    targets = r.json().get("targets", [])
    if not targets:
        return None
    return targets[0]

L1 = {}
for t in TARGETS:
    print(f"\n=== {t['locus']}  {t['gene']} ===")
    for acc, org in t['orthologs']:
        try:
            tgt = chembl_target_for_uniprot(acc)
        except Exception as e:
            tgt = None
            print(f"  {acc:8s} ({org:25s}): FAIL {type(e).__name__}: {e}")
            continue
        if tgt:
            tid = tgt.get("target_chembl_id")
            tname = tgt.get("pref_name")
            torg = tgt.get("organism")
            print(f"  {acc:8s} ({org:25s}): ChEMBL target {tid}  {tname[:50] if tname else '?':50s}  org={torg}")
            L1.setdefault(t['locus'], []).append({
                "uniprot": acc, "organism_label": org,
                "chembl_target_id": tid, "target_pref_name": tname,
                "target_organism": torg,
            })
        else:
            print(f"  {acc:8s} ({org:25s}): no ChEMBL target record")
        time.sleep(0.3)

In [ ]:
# Cell 4 — Layer 2: (target_chembl_id, molecule_chembl_id) → activities.
# For each (Mycoplasma target, compound) pair found above, query
# activity records. This is the corrected query — only returns assays
# specifically of THIS compound on THIS Mycoplasma protein.
L2 = {}
for t in TARGETS:
    print(f"\n=== activities: {t['gene']} ({t['drug_name']}) ===")
    drug_id = t['drug_chembl_id']
    if not drug_id:
        print(f"  no compound CHEMBL ID — skip")
        continue
    for entry in (L1.get(t['locus']) or []):
        tid = entry.get("chembl_target_id")
        if not tid:
            continue
        try:
            r = requests.get(
                f"{CHEMBL}/activity.json",
                params={"target_chembl_id": tid,
                        "molecule_chembl_id": drug_id,
                        "limit": "500"},
                headers=UA, timeout=30,
            )
            r.raise_for_status()
            data = r.json()
        except Exception as e:
            print(f"  {tid} x {drug_id}: FAIL {type(e).__name__}: {e}")
            continue
        acts = data.get("activities", [])
        print(f"  {tid} ({entry['organism_label'][:25]:25s}) x {drug_id}: {len(acts)} records")
        for a in acts[:5]:
            print(f"    {(a.get('standard_type') or '?'):8s}  "
                  f"{a.get('standard_value','?')} {a.get('standard_units','?')}  "
                  f"assay={a.get('assay_chembl_id','?')}  "
                  f"organism={a.get('assay_organism','?')}")
        L2.setdefault(t['locus'], []).append({
            "chembl_target_id": tid,
            "organism_label": entry['organism_label'],
            "compound_chembl_id": drug_id,
            "n_activities": len(acts),
            "activities": [{
                "standard_type": a.get("standard_type"),
                "standard_value": a.get("standard_value"),
                "standard_units": a.get("standard_units"),
                "assay_organism": a.get("assay_organism"),
                "assay_chembl_id": a.get("assay_chembl_id"),
                "document_chembl_id": a.get("document_chembl_id"),
            } for a in acts],
        })
        time.sleep(0.3)

In [ ]:
# Cell 5 — Universe check: enumerate ALL Mycoplasma targets in ChEMBL.
# Tells us the ceiling on what's queryable for any Mycoplasma protein.
print("=== ChEMBL targets where organism contains 'Mycoplasma' ===")
L3 = []
offset = 0
while True:
    r = requests.get(
        f"{CHEMBL}/target.json",
        params={"organism__icontains": "Mycoplasma",
                "limit": "500", "offset": str(offset)},
        headers=UA, timeout=30,
    )
    if r.status_code != 200:
        print(f"  FAIL: status {r.status_code}")
        break
    data = r.json()
    tgts = data.get("targets", [])
    if not tgts:
        break
    for tgt in tgts:
        L3.append({
            "target_chembl_id": tgt.get("target_chembl_id"),
            "pref_name": tgt.get("pref_name"),
            "organism": tgt.get("organism"),
            "target_type": tgt.get("target_type"),
        })
    if not data.get("page_meta", {}).get("next"):
        break
    offset += 500

print(f"\ntotal ChEMBL targets where organism contains 'Mycoplasma': {len(L3)}")
from collections import Counter
org_counter = Counter(x['organism'] for x in L3 if x.get('organism'))
print("\norganism breakdown:")
for org, n in org_counter.most_common(20):
    print(f"  {n:4d}  {org}")
ttype_counter = Counter(x['target_type'] for x in L3 if x.get('target_type'))
print("\ntarget type breakdown:")
for tt, n in ttype_counter.most_common(10):
    print(f"  {n:4d}  {tt}")

In [ ]:
# Cell 6 — consolidate + auto-push.
from pathlib import Path
out = {
    "targets": TARGETS,
    "layer_1_uniprot_to_chembl_target": L1,
    "layer_2_target_x_compound_activities": L2,
    "layer_3_universe_mycoplasma_targets": L3,
}
out_path = Path("outputs/toxicity/path_e_mic_lookup_v2.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(out, indent=2, default=str))
print(f"wrote {out_path} ({out_path.stat().st_size} bytes)")

pat = os.environ.get("GITHUB_PAT", "").strip()
if not pat: raise SystemExit("GITHUB_PAT not set")
_run(["git", "config", "user.email", "cell-sim-bot@noreply.local"])
_run(["git", "config", "user.name", "cell-sim-bot"])
_run(["git", "add", "-f", str(out_path)])
status = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True)
if status.stdout.strip():
    _run(["git", "commit", "-m",
          "Session 27 v2: corrected ChEMBL Mycoplasma activity query"])
    remote = f"https://{pat}@github.com/Nikku03/cell.git"
    _run(["git", "push", remote, BRANCH])
    print("\npush complete.")
else:
    print("nothing changed")